In [35]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from datasets import Dataset
import pandas as pd
from pathlib import Path
import torch

DATA_DIR = Path.cwd().parent.parent / "proj_data" / "data" / "subtask2"
TRAIN_DIR = Path.cwd().parent.parent / "proj_data" / "data" / "subtask2" / "train"
DEV_DIR = Path.cwd().parent.parent / "proj_data" / "data" / "subtask2" / "dev"
csvs = ["amh.csv", "arb.csv", "deu.csv", "eng.csv", "hau.csv", "ita.csv", "spa.csv", "urd.csv", "zho.csv"]
train_data = pd.DataFrame()
dev_data = pd.DataFrame()
for csv in csvs:
    tr_pth = TRAIN_DIR / csv
    tmp = pd.read_csv(tr_pth)
    tmp["source"] = csv.split(".")[0]
    train_data = pd.concat([train_data, tmp], ignore_index=True)

    dv_pth = DEV_DIR / csv
    tmp = pd.read_csv(dv_pth)
    tmp["source"] = csv.split(".")[0]
    dev_data = pd.concat([dev_data, tmp], ignore_index=True)

tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
base_model = AutoModel.from_pretrained("xlm-roberta-base")
model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=5,                # one for each type
    problem_type="multi_label_classification"
)
print("Tokenizer OK:", tok is not None, "Model OK:", model is not None)
print("Train data", train_data.columns, train_data.shape)
print("Dev data", dev_data.columns, dev_data.shape)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizer OK: True Model OK: True
Train data Index(['id', 'text', 'political', 'racial/ethnic', 'religious',
       'gender/sexual', 'other', 'source'],
      dtype='object') (29987, 8)
Dev data Index(['id', 'text', 'political', 'racial/ethnic', 'religious',
       'gender/sexual', 'other', 'source'],
      dtype='object') (1496, 8)


In [40]:
def tokenize(batch):
    return tok(
        batch["text"],
        padding="max_length",       # pad to a fixed size
        truncation=True,            # truncate long text
        max_length=128,             # or 256 depending on GPU memory
        return_tensors="pt"
    )
label_cols = ['political', 'racial/ethnic', 'religious',
       'gender/sexual', 'other',]
train_data["labels"] = train_data[label_cols].values.tolist()
train_dataset = Dataset.from_pandas(train_data)
tokenized_train = train_dataset.map(tokenize, batched=True)
# tokenized_train = tokenized_train.rename_column("text", "input_text")
# tokenized_train = tokenized_train.map(
#     lambda x: {"labels": [float(v) for v in x["labels"]]}
# )
# tokenized_train.set_format(
#     type="torch",
#     columns=["input_ids", "attention_mask", "labels"]
# )
def cast_labels(example):
    example["labels"] = torch.tensor(example["labels"], dtype=torch.float)
    return example

# tokenized_train = tokenized_train.map(cast_labels)
tokenized_train = tokenized_train.map(
    lambda x: {"labels": [float(v) for v in x["labels"]]}
)
tokenized_train = tokenized_train.map(
    lambda x: {"labels": torch.tensor(x["labels"], dtype=torch.float)}
)
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 29987/29987 [00:01<00:00, 28422.27 examples/s]


In [41]:
print(tokenized_train[0]["labels"].dtype)
print(type(tokenized_train[0]["labels"][0]))

torch.int64
<class 'torch.Tensor'>


In [33]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = (1 / (1 + np.exp(-logits))) > 0.5
    return {"macro_f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    compute_metrics=compute_metrics
)

trainer.train()

RuntimeError: result type Float can't be cast to the desired output type Long